## 1. Setup & Imports

In [ ]:
# Install dependencies if needed (uncomment if running on Colab/fresh env)
# !pip install torch torchvision matplotlib numpy --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import time

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Part 1 – PrunableLinear Layer


Each weight `w_ij` gets a learnable **gate score** `s_ij` of the same shape.  
During the forward pass:

```
gate_ij        = sigmoid(s_ij)          ∈ (0, 1)
pruned_weight  = w_ij  ×  gate_ij
output         = pruned_weight @ x  +  bias
```

When `gate_ij → 0`, the connection is effectively **pruned**.  
PyTorch autograd handles gradient flow through both `weight` and `gate_scores` automatically.


In [ ]:
class PrunableLinear(nn.Module):
    """
    A custom Linear layer with learnable gate parameters.

    Each weight w_ij has a paired gate_score s_ij (same shape).
    Effective weight = w_ij * sigmoid(s_ij).

    When sigmoid(s_ij) → 0, the weight is effectively pruned.
    Gradients flow through both `weight` and `gate_scores` via autograd.
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        # ── Standard parameters (same as nn.Linear) ──
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

        # ── Gate scores: same shape as weight ──
        # Init to 1.0 → sigmoid(1) ≈ 0.73, so gates start mostly open.
        # The optimizer will learn to close (zero) unimportant ones.
        self.gate_scores = nn.Parameter(torch.ones(out_features, in_features))

        # Kaiming init for weight (matches nn.Linear default)
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Step 1 – Convert raw scores → gates ∈ (0, 1)
        gates = torch.sigmoid(self.gate_scores)          # (out_features, in_features)

        # Step 2 – Element-wise multiply: prune unimportant weights
        pruned_weights = self.weight * gates              # (out_features, in_features)

        # Step 3 – Standard linear transform (gradients flow through both paths)
        return F.linear(x, pruned_weights, self.bias)

    # ── Utility methods ───────────────────────────────
    def get_gates(self) -> torch.Tensor:
        """Return current gate values, detached from computation graph."""
        return torch.sigmoid(self.gate_scores).detach()

    def sparsity_fraction(self, threshold: float = 1e-2) -> float:
        """Fraction of weights whose gate < threshold (effectively pruned)."""
        gates = self.get_gates()
        return (gates < threshold).float().mean().item()

    def extra_repr(self) -> str:
        return f"in={self.in_features}, out={self.out_features}"


# ── Quick sanity check ────────────────────────────────────────────────────────
print("=== PrunableLinear Sanity Check ===")
layer = PrunableLinear(4, 3)
x_test = torch.randn(2, 4, requires_grad=True)
y_test = layer(x_test)
loss_test = y_test.sum()
loss_test.backward()

print(f"Input shape  : {x_test.shape}")
print(f"Output shape : {y_test.shape}")
print(f"weight.grad  : {'✓ flowing' if layer.weight.grad is not None else '✗ None'}")
print(f"gate .grad   : {'✓ flowing' if layer.gate_scores.grad is not None else '✗ None'}")
print(f"Initial gate values (sample): {layer.get_gates()[0, :4].numpy().round(3)}")
print(f"Initial sparsity (gate<0.01) : {layer.sparsity_fraction():.1%}")


## 3. Network Architecture

A feed-forward network for CIFAR-10.  
All linear layers replaced with `PrunableLinear`.

```
Input: 3×32×32 → flatten → 3072
  PrunableLinear(3072 → 1024) + BatchNorm + ReLU + Dropout(0.3)
  PrunableLinear(1024 →  512) + BatchNorm + ReLU + Dropout(0.3)
  PrunableLinear( 512 →  256) + BatchNorm + ReLU
  PrunableLinear( 256 →   10)   ← logits (10 CIFAR classes)
```


In [ ]:
class SelfPruningNet(nn.Module):
    """Feed-forward network with self-pruning via PrunableLinear layers."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            PrunableLinear(3072, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            PrunableLinear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            PrunableLinear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            PrunableLinear(256, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)   # Flatten: (B, 3, 32, 32) → (B, 3072)
        return self.net(x)

    def prunable_layers(self):
        """Generator yielding all PrunableLinear sub-modules."""
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                yield m

    def sparsity_loss(self) -> torch.Tensor:
        """
        L1 norm of ALL gate values across every PrunableLinear layer.
        Minimising this drives gates toward 0 (pruning weights).
        """
        all_gates = [torch.sigmoid(l.gate_scores) for l in self.prunable_layers()]
        return torch.cat([g.view(-1) for g in all_gates]).sum()

    def overall_sparsity(self, threshold: float = 1e-2) -> float:
        """Percentage of total weights effectively pruned (gate < threshold)."""
        total, pruned = 0, 0
        for layer in self.prunable_layers():
            g = layer.get_gates()
            pruned += (g < threshold).sum().item()
            total  += g.numel()
        return 100.0 * pruned / total if total > 0 else 0.0

    def all_gate_values(self) -> np.ndarray:
        """Flat numpy array of every gate value — for histogram analysis."""
        vals = [l.get_gates().cpu().numpy().ravel() for l in self.prunable_layers()]
        return np.concatenate(vals)

    def count_parameters(self) -> dict:
        total = sum(p.numel() for p in self.parameters())
        weight_params = sum(l.weight.numel() for l in self.prunable_layers())
        gate_params   = sum(l.gate_scores.numel() for l in self.prunable_layers())
        return {"total": total, "weights": weight_params, "gates": gate_params}


# ── Print model summary ───────────────────────────────────────────────────────
model_demo = SelfPruningNet()
print("=== SelfPruningNet Architecture ===")
print(model_demo)
params = model_demo.count_parameters()
print(f"\nTotal parameters : {params['total']:,}")
print(f"  Weight params  : {params['weights']:,}")
print(f"  Gate  params   : {params['gates']:,}")
print(f"  (gates add ~{100*params['gates']/params['total']:.1f}% overhead)")


## 4. Part 2 – Sparsity Regularization Loss



In [ ]:
# Demonstrate sparsity loss behaviour
demo_net = SelfPruningNet()

# Compute sparsity loss before any training
with torch.no_grad():
    initial_sparse_loss = demo_net.sparsity_loss().item()

total_gates = sum(l.gate_scores.numel() for l in demo_net.prunable_layers())
print(f"Total gate parameters : {total_gates:,}")
print(f"Initial sparsity loss : {initial_sparse_loss:,.1f}")
print(f"Mean gate value (init): {initial_sparse_loss / total_gates:.4f}  (sigmoid(1) ≈ 0.731)")
print()

# Show effect of different lambda on total loss magnitude
ce_example = torch.tensor(2.3)   # typical initial cross-entropy for 10 classes
for lam in [1e-5, 1e-4, 5e-4]:
    total = ce_example + lam * initial_sparse_loss
    print(f"  λ={lam:.0e}  →  total loss ≈ {total:.3f}  "
          f"(CE={ce_example:.2f}, sparse_term={lam*initial_sparse_loss:.3f})")


## 5. Part 3 – Data Loading (CIFAR-10)

In [ ]:
def get_cifar10_loaders(batch_size: int = 256):
    """Download CIFAR-10 and return train/test DataLoaders."""
    # Training: random flip + crop for augmentation
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    # Test: only normalize (no augmentation)
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])

    train_ds = datasets.CIFAR10(root="./data", train=True,
                                download=True, transform=transform_train)
    test_ds  = datasets.CIFAR10(root="./data", train=False,
                                download=True, transform=transform_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader


train_loader, test_loader = get_cifar10_loaders(batch_size=256)
print(f"Train batches : {len(train_loader)}  ({len(train_loader.dataset):,} samples)")
print(f"Test  batches : {len(test_loader)}  ({len(test_loader.dataset):,} samples)")

# Show sample images
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
mean = np.array([0.4914, 0.4822, 0.4465])
std  = np.array([0.2023, 0.1994, 0.2010])
for i, ax in enumerate(axes.ravel()):
    img = imgs[i].permute(1,2,0).numpy()
    img = np.clip(img * std + mean, 0, 1)
    ax.imshow(img)
    ax.set_title(CLASSES[labels[i]], fontsize=8)
    ax.axis("off")
plt.suptitle("CIFAR-10 Sample Images", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Training & Evaluation Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, lam, device):
    """
    One full training epoch.
    Loss = CrossEntropy(logits, targets) + λ × SparsityLoss(gates)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        logits = model(inputs)

        # ── Classification loss ──
        cls_loss = F.cross_entropy(logits, targets)

        # ── Sparsity loss (L1 of all gate values) ──
        sparse_loss = model.sparsity_loss()

        # ── Total loss ──
        loss = cls_loss + lam * sparse_loss
        loss.backward()

        # Gradient clipping for training stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        preds = logits.argmax(dim=1)
        correct += preds.eq(targets).sum().item()
        total += inputs.size(0)

    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    """Compute test accuracy."""
    model.eval()
    correct, total = 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        preds = model(inputs).argmax(dim=1)
        correct += preds.eq(targets).sum().item()
        total += inputs.size(0)
    return 100.0 * correct / total


def run_experiment(lam: float, train_loader, test_loader, device, epochs: int = 30):
    """
    Train a fresh SelfPruningNet with a given λ for `epochs` epochs.
    Returns a dict with final accuracy, sparsity, and gate values.
    """
    print(f"\n{'='*60}")
    print(f"  Training  λ = {lam}")
    print(f"{'='*60}")

    model = SelfPruningNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    # CosineAnnealing: smoothly decays LR to near-zero over `epochs`
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "train_acc": [], "test_acc": [], "sparsity": []}

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, lam, device)
        test_acc = evaluate(model, test_loader, device)
        sparsity = model.overall_sparsity()
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        history["sparsity"].append(sparsity)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs}  │  "
                  f"Loss {train_loss:.4f}  │  "
                  f"TrainAcc {train_acc:.1f}%  │  "
                  f"TestAcc {test_acc:.1f}%  │  "
                  f"Sparsity {sparsity:.1f}%  │  "
                  f"{time.time()-t0:.1f}s")

    final_acc      = evaluate(model, test_loader, device)
    final_sparsity = model.overall_sparsity()
    gate_values    = model.all_gate_values()

    print(f"\n  ✅ Final Test Accuracy  : {final_acc:.2f}%")
    print(f"  ✅ Final Sparsity Level : {final_sparsity:.2f}%")

    return {
        "lambda":        lam,
        "test_accuracy": final_acc,
        "sparsity":      final_sparsity,
        "gate_values":   gate_values,
        "history":       history,
        "model":         model,
    }

print("Training functions defined ✓")


## 7. Run Experiments (3 × λ values)

We compare **three λ values**:
- `1e-5` — **Low** sparsity pressure (barely prunes)
- `1e-4` — **Medium** sparsity pressure (balanced)
- `5e-4` — **High** sparsity pressure (aggressive pruning)


In [ ]:
EPOCHS  = 30      # Increase to 50+ for better accuracy
LAMBDAS = [1e-5, 1e-4, 5e-4]

results = []
for lam in LAMBDAS:
    res = run_experiment(lam, train_loader, test_loader, device, EPOCHS)
    results.append(res)


## 8. Results Table

In [ ]:
print("\n" + "="*55)
print("  RESULTS SUMMARY")
print("="*55)
print(f"  {'Lambda':<12} {'Test Accuracy':>15} {'Sparsity (%)':>14}")
print(f"  {'-'*12} {'-'*15} {'-'*14}")
for r in results:
    marker = " ← best acc" if r['test_accuracy'] == max(x['test_accuracy'] for x in results) else ""
    print(f"  {r['lambda']:<12} {r['test_accuracy']:>14.2f}%{r['sparsity']:>13.2f}%{marker}")
print("="*55)

# ── Training curve plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ["#E74C3C", "#3498DB", "#2ECC71"]

for ax, res, color in zip(axes, results, colors):
    h = res["history"]
    epochs_range = range(1, len(h["test_acc"]) + 1)
    ax.plot(epochs_range, h["train_acc"], color=color, alpha=0.5, label="Train Acc")
    ax.plot(epochs_range, h["test_acc"],  color=color, linewidth=2, label="Test Acc")
    ax2 = ax.twinx()
    ax2.plot(epochs_range, h["sparsity"], color="gray",
             linestyle="--", linewidth=1.5, label="Sparsity %")
    ax2.set_ylabel("Sparsity (%)", color="gray")
    ax2.tick_params(axis='y', labelcolor='gray')
    ax.set_title(f"λ = {res['lambda']}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.legend(loc="lower right", fontsize=8)
    ax.set_ylim(0, 100)

plt.suptitle("Training Curves: Accuracy & Sparsity vs Epoch", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 9. Gate Distribution Plot

A **successful** self-pruning model shows a **bimodal** distribution:
- 🔴 Large spike near **0** — pruned weights
- 🔵 Cluster closer to **1** — active (important) weights


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
colors    = ["#E74C3C", "#3498DB", "#2ECC71"]

for ax, res, color in zip(axes, results, colors):
    gates = res["gate_values"]
    pruned_frac = (gates < 0.01).mean() * 100

    ax.hist(gates, bins=100, color=color, alpha=0.85,
            edgecolor="white", linewidth=0.2)
    ax.axvline(x=0.01, color="black", linestyle="--",
               linewidth=1.5, label=f"Threshold (0.01)")

    ax.set_title(
        f"λ = {res['lambda']}
"
        f"Test Acc: {res['test_accuracy']:.1f}%  │  "
        f"Sparsity: {res['sparsity']:.1f}%",
        fontsize=10, fontweight="bold"
    )
    ax.set_xlabel("Gate Value  g = sigmoid(s)", fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.legend(fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Annotate fraction pruned
    ax.text(0.6, ax.get_ylim()[1]*0.85,
            f"{pruned_frac:.1f}% pruned",
            fontsize=10, color="black",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="gray"))

plt.suptitle("Gate Value Distributions After Training
"
             "(Bimodal = successful sparsity learning)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("gate_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved → gate_distributions.png")
